<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/excercise/RAG_supabase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 다양한 파일 타입(PDF, Text, CSV)을 수집하여 Supabase(PostgreSQL + pgvector)와 지식 그래프를 활용해 RDB, VectorDB, GraphDB 구축부터 질문 라우팅(Routing)까지 단계별로 구성한 Python 코드 가이드

<img src="https://miro.medium.com/v2/resize:fit:750/format:webp/1*auq3pjIxRDEO0DMz8xgRrg.png">

In [ ]:
# @title Phase 1: 개발 환경 및 Supabase DB 초기화
!pip install langchain langchain-community langchain-groq langchain-google-genai langchain-huggingface supabase pandas pypdf networkx -q

Supabase SQL Setup & Run

'''
-- pgvector 확장 활성화
CREATE EXTENSION IF NOT EXISTS vector;

-- 1. RDB: CSV 메타데이터 저장용
CREATE TABLE IF NOT EXISTS structured_data (
    id SERIAL PRIMARY KEY,
    entity_name TEXT,
    category TEXT,
    metric_value NUMERIC,
    metadata JSONB
);

-- 2. VectorDB: PDF/Text 벡터 저장용 (all-MiniLM-L6-v2차원: 384)
CREATE TABLE IF NOT EXISTS document_vectors (
    id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    content TEXT,
    metadata JSONB,
    embedding VECTOR(384)
);

-- 검색용 RPC 함수 정의
CREATE OR REPLACE FUNCTION match_documents (
  query_embedding VECTOR(384),
  match_threshold FLOAT,
  match_count INT
)
RETURNS TABLE (
  id UUID,
  content TEXT,
  metadata JSONB,
  similarity FLOAT
)
LANGUAGE plpgsql
AS $$
BEGIN
  RETURN QUERY
  SELECT
    document_vectors.id,
    document_vectors.content,
    document_vectors.metadata,
    1 - (document_vectors.embedding <=> query_embedding) AS similarity
  FROM document_vectors
  WHERE 1 - (document_vectors.embedding <=> query_embedding) > match_threshold
  ORDER BY document_vectors.embedding <=> query_embedding
  LIMIT match_count;
END;
$$;

-- 3. GraphDB: 엔티티 간 관계 저장용 (Supabase 테이블 방식 Graph)
CREATE TABLE IF NOT EXISTS graph_nodes (
    node_id TEXT PRIMARY KEY,
    label TEXT,
    properties JSONB
);

CREATE TABLE IF NOT EXISTS graph_edges (
    id SERIAL PRIMARY KEY,
    source_id TEXT REFERENCES graph_nodes(node_id),
    target_id TEXT REFERENCES graph_nodes(node_id),
    relation TEXT,
    properties JSONB
);
'''

In [ ]:
# @title Generate sample data
!pip install reportlab

import os
import pandas as pd
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

# 1. 샘플 CSV 생성 (RDB 저장용)
csv_data = {
    "id": [1, 2, 3],
    "name": ["Project Alpha", "Project Beta", "Project Gamma"],
    "category": ["AI", "Cloud", "Data"],
    "value": [15000.0, 23000.0, 8500.0]
}
df = pd.DataFrame(csv_data)
df.to_csv("sample.csv", index=False, encoding="utf-8")
print("📁 sample.csv 생성 완료")

# 2. 샘플 Text 생성 (VectorDB 저장용)
text_content = """
Supabase는 오픈소스 Firebase 대안 플랫폼입니다.
PostgreSQL을 기반으로 구축되었으며 pgvector 확장을 통해 벡터 임베딩 저장 및 유사도 검색을 지원합니다.
LLM 어플리케이션 개발 시 VectorDB와 RDB 역할을 동시에 수행할 수 있어 매우 유용합니다.
"""
with open("sample.txt", "w", encoding="utf-8") as f:
    f.write(text_content.strip())
print("📁 sample.txt 생성 완료")

# 3. 샘플 PDF 생성 (VectorDB 저장용)
def create_pdf(filename):
    c = canvas.Canvas(filename, pagesize=letter)
    c.drawString(100, 750, "Project Alpha Overview Document")
    c.drawString(100, 730, "Objective: Build a Next-Gen RAG Architecture.")
    c.drawString(100, 710, "Technologies: LangChain, Supabase, Hugging Face, Gemini, ChatGroq.")
    c.save()

create_pdf("sample.pdf")
print("📁 sample.pdf 생성 완료")

📁 sample.csv 생성 완료
📁 sample.txt 생성 완료
📁 sample.pdf 생성 완료


In [ ]:
# @title Phase 2: Indexing Pipeline (데이터 로드 및 DB 분할 저장)

# ==========================================
# 2단계: 수파베이스 접속 정보 설정
# ==========================================
# 수파베이스 대시보드의 [Project Settings] -> [API]에서 확인한 정보를 입력합니다.

from google.colab import userdata
SUPABASE_KEY = userdata.get('supakey')

# 접속이 되는 지 사전 확인
import socket
from urllib.parse import urlparse

SUPABASE_URL = "https://qctwuimxnppnzmjcnluc.supabase.co"
hostname = urlparse(SUPABASE_URL).hostname

try:
    ip = socket.gethostbyname(hostname)
    print(f"DNS 조회 성공: {ip}")
except socket.gaierror as e:
    print(f"DNS 조회 실패: {e}")

DNS 조회 성공: 104.18.38.10


In [ ]:
# @title Connect Supabase DB & import HuggingFaceEmbeddings
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

# 수파베이스 공식 파이썬 클라이언트 객체를 생성
supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 로컬 HuggingFace 임베딩 모델 소환
embed_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_12643/440671835.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import SupabaseVectorStore


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# @title Phase 3: Migrate sample data to Supabase

import os
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import SupabaseVectorStore

# ----------------------------------------------------
# 1. Text & PDF 로드 -> VectorDB (document_vectors) 저장
# ----------------------------------------------------
def load_and_index_documents(file_paths: list):
    all_chunks = []
    splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

    for file_path in file_paths:
        if file_path.endswith(".pdf"):
            loader = PyPDFLoader(file_path)
        elif file_path.endswith(".txt"):
            loader = TextLoader(file_path, encoding="utf-8")
        else:
            continue

        docs = loader.load()
        chunks = splitter.split_documents(docs)
        all_chunks.extend(chunks)

    # Supabase VectorStore로 임베딩 변환 및 저장 (embed_model, supabase_client 활용)
    vector_store = SupabaseVectorStore.from_documents(
        documents=all_chunks,
        embedding=embed_model,
        client=supabase_client,
        table_name="document_vectors",
        query_name="match_documents"
    )
    print(f"✅ [VectorDB] 총 {len(all_chunks)}개 청크 저장 완료 (sample.txt, sample.pdf)")


# ----------------------------------------------------
# 2. CSV 로드 -> RDB (structured_data) 저장
# ----------------------------------------------------
def load_and_index_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    records = []

    for _, row in df.iterrows():
        records.append({
            "entity_name": str(row["name"]),
            "category": str(row["category"]),
            "metric_value": float(row["value"]),
            "metadata": row.to_dict()
        })

    # Supabase 테이블에 배치 삽입 (supabase_client 활용)
    supabase_client.table("structured_data").insert(records).execute()
    print(f"✅ [RDB] 총 {len(records)}개 행 저장 완료 (sample.csv)")


# ----------------------------------------------------
# 3. Knowledge Triples -> GraphDB (graph_nodes, graph_edges) 저장
# ----------------------------------------------------
def load_and_index_graph(triples: list):
    for src, rel, tgt in triples:
        # 노드 등록 (UPSERT) (supabase_client 활용)
        supabase_client.table("graph_nodes").upsert([
            {"node_id": src, "label": "Project"},
            {"node_id": tgt, "label": "Technology"}
        ]).execute()

        # 엣지(관계) 등록 (supabase_client 활용)
        supabase_client.table("graph_edges").insert({
            "source_id": src,
            "target_id": tgt,
            "relation": rel
        }).execute()

    print(f"✅ [GraphDB] 총 {len(triples)}개 노드-관계 저장 완료")


# ----------------------------------------------------
# 실행 파이프라인
# ----------------------------------------------------
if __name__ == "__main__":
    print("🚀 Supabase 데이터 인덱싱 시작...\n")

    # 1. PDF 및 Text 파일 VectorDB 저장
    load_and_index_documents(["sample.txt", "sample.pdf"])

    # 2. CSV 파일 RDB 저장
    load_and_index_csv("sample.csv")

    # 3. 그래프 데이터 GraphDB 저장
    sample_triples = [
        ("Project Alpha", "USES_TECH", "LangChain"),
        ("Project Alpha", "STORED_IN", "Supabase"),
        ("Supabase", "HAS_EXTENSION", "pgvector")
    ]
    load_and_index_graph(sample_triples)

    print("\n🎉 모든 데이터가 성공적으로 Supabase DB에 저장되었습니다!")

🚀 Supabase 데이터 인덱싱 시작...

✅ [VectorDB] 총 2개 청크 저장 완료 (sample.txt, sample.pdf)
✅ [RDB] 총 3개 행 저장 완료 (sample.csv)
✅ [GraphDB] 총 3개 노드-관계 저장 완료

🎉 모든 데이터가 성공적으로 Supabase DB에 저장되었습니다!


#### Supabase 대시보드 > Table Editor에 접속하면 데이터 저장 여부를 바로 확인
- document_vectors 테이블: content 컬럼과 384차원 실수가 채워진 embedding 벡터 수신 확인
- structured_data 테이블: Project Alpha, Project Beta 등의 CSV 컬럼 값 등록 확인
- graph_nodes & graph_edges 테이블: Project Alpha $\rightarrow$ LangChain 관계 트리 연결 확인

In [ ]:
# @title # LLM 선언(API 키가 설정되었는지 확인 후 객체 생성 (실패 시 예외 처리))
from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata
import os

# Colab 비밀번호(userdata)에서 API 키 가져오기 및 환경 변수 설정
try:
    groq_key = userdata.get('groq')
    if groq_key:
        os.environ["GROQ_API_KEY"] = groq_key
except Exception:
    pass

try:
    gemini_key = userdata.get('Gemini-API-Key')
    if gemini_key:
        os.environ["GOOGLE_API_KEY"] = gemini_key
except Exception:
    pass



try:
    llm_groq = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)
except Exception as e:
    print(f"Groq LLM 초기화 실패 (API 키 확인 필요): {e}")
    llm_groq = None

try:
    llm_gemini = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.1)
except Exception as e:
    print(f"Gemini LLM 초기화 실패 (API 키 확인 필요): {e}")
    llm_gemini = None


In [ ]:
# @title Phase 5: Router & Retrieve (질문 분석, 라우팅 및 조회)

# 질문 라우팅을 위한 Structured Output 구조체
class RouteQuery(BaseModel):
    destination: Literal["rdb", "vectordb", "graphdb"] = Field(
        ...,
        description="Select DB type based on the query: 'rdb' for numeric/tabular/stats, 'vectordb' for textual details/unstructured info, 'graphdb' for relationship/network/connection queries."
    )

def route_and_retrieve(query: str, selected_llm=None) -> dict:
    # 기본 LLM 설정
    if selected_llm is None:
        selected_llm = llm_gemini if llm_gemini else llm_groq

    if not selected_llm:
        raise ValueError("사용 가능한 LLM 인스턴스가 없습니다. API 키 설정을 확인해 주세요.")

    # 1. Routing 단계
    router_llm = selected_llm.with_structured_output(RouteQuery)
    route_result = router_llm.invoke(f"Analyze the user query and pick the best database destination:\nQuery: {query}")
    target_db = route_result.destination

    print(f"ፇ Router Decision: [{target_db.upper()}] for query: '{query}'")

    retrieved_context = ""

    # 2. Retrieval 단계
    if target_db == "vectordb":
        # supabase_client와 embed_model 변수는 이전 셀에서 정의한 것을 활용합니다.
        vector_store = SupabaseVectorStore(
            client=supabase_client,
            embedding=embed_model,
            table_name="document_vectors",
            query_name="match_documents"
        )
        docs = vector_store.similarity_search(query, k=3)
        retrieved_context = "\n---\n".join([d.page_content for d in docs])

    elif target_db == "rdb":
        response = supabase_client.table("structured_data").select("*").limit(10).execute()
        retrieved_context = str(response.data)

    elif target_db == "graphdb":
        nodes = supabase_client.table("graph_nodes").select("*").execute().data
        edges = supabase_client.table("graph_edges").select("*").execute().data
        retrieved_context = f"Nodes: {nodes}\nEdges: {edges}"

    return {"target_db": target_db, "context": retrieved_context}

In [ ]:
# @title Phase 5: Generation & Pipeline Execution (최종 파이프라인 구축)

def run_pipeline(user_query: str, model_type: str = "groq"):
    llm = llm_groq if model_type == "groq" else llm_gemini

    # 1) Routing & Retrieval
    retrieved = route_and_retrieve(user_query, selected_llm=llm)

    # 2) RAG Final Generation Prompt
    generation_prompt = f"""You are a knowledge assistant. Answer the question based ONLY on the retrieved data context provided below.
If the answer cannot be determined from the context, state that you don't have enough information.

[Retrieved Database Source]: {retrieved['target_db'].upper()}
[Retrieved Context]:
{retrieved['context']}

[User Question]:
{user_query}

[Answer (in Korean)]:"""

    response = llm.invoke(generation_prompt)

    print("================ [ Final Answer ] ================")
    print(response.content)
    print("==================================================\n")


# ----------------------------------------------------
# 3. 실습 테스트 질문 실행 (실제 인덱스 데이터 기준)
# ----------------------------------------------------
if __name__ == "__main__":
    print("🤖 LLM Multi-DB RAG Pipeline Execution Test\n")

    # [Test 1] VectorDB 라우팅 테스트 (sample.txt & sample.pdf 내용)
    print("--- Test 1: Document Context (VectorDB) ---")
    run_pipeline("Supabase가 LLM 애플리케이션 개발에서 어떤 장점이 있는지 문서 내용을 바탕으로 설명해줘.", model_type="groq")

    # [Test 2] RDB 라우팅 테스트 (sample.csv 내용)
    print("--- Test 2: Structured Tabular Data (RDB) ---")
    run_pipeline("현재 등록되어 있는 프로젝트들의 이름, 카테고리, value 값을 확인하고 전체 value 합계를 계산해줘.", model_type="gemini")

    # [Test 3] GraphDB 라우팅 테스트 (Knowledge Triples 내용)
    print("--- Test 3: Entity Relationships (GraphDB) ---")
    run_pipeline("Project Alpha는 어떤 기술(Technology)들과 연결되어 있고, Supabase는 어떤 확장 기능을 가지고 있는지 관계를 알려줘.", model_type="groq")

🤖 LLM Multi-DB RAG Pipeline Execution Test

--- Test 1: Document Context (VectorDB) ---
ፇ Router Decision: [VECTORDB] for query: 'Supabase가 LLM 애플리케이션 개발에서 어떤 장점이 있는지 문서 내용을 바탕으로 설명해줘.'
================ [ Final Answer ] ================
죄송합니다. 제공된 문서 내용에는 Supabase와 LLM 애플리케이션 개발에 관한 정보가 포함되어 있지 않으므로, 해당 질문에 대한 답변을 제공할 수 없습니다.

--- Test 2: Structured Tabular Data (RDB) ---


ፇ Router Decision: [RDB] for query: '현재 등록되어 있는 프로젝트들의 이름, 카테고리, value 값을 확인하고 전체 value 합계를 계산해줘.'
================ [ Final Answer ] ================
[{'type': 'text', 'text': '제공된 데이터에 등록된 프로젝트의 이름, 카테고리, Value 값은 다음과 같습니다. 데이터 내에 동일한 프로젝트 정보가 중복되어 존재하므로, 고유 프로젝트 기준과 전체 데이터 기준의 합계를 각각 계산해 드립니다.\n\n### 1. 등록된 프로젝트 정보 (고유 기준)\n* **Project Alpha** (카테고리: AI) — Value: 15,000.0\n* **Project Beta** (카테고리: Cloud) — Value: 23,000.0\n* **Project Gamma** (카테고리: Data) — Value: 8,500.0\n\n* **고유 프로젝트 Value 합계**: **46,500.0** (15,000.0 + 23,000.0 + 8,500.0)\n\n---\n\n### 2. 전체 데이터 기준 (중복 포함 총 6개 항목)\n데이터 리스트에 동일한 프로젝트가 각각 2번씩 등록되어 있습니다.\n* Project Alpha (AI): 15,000.0 (2개)\n* Project Beta (Cloud): 23,000.0 (2개)\n* Project Gamma (Data): 8,500.0 (2개)\n\n* **전체 데이터 Value 합계**: **93,000.0** (15,000.0 × 2 + 23,000.0 × 2 + 8,500.0 × 2)', 'extras': {'signature': 'Eu0jCuojAWkUfRND2CXmMbRBx7DfKzDd+0FU62QUuGiOO9g9eg2RtDSJ9QGO15IGq/X3uPoGTd6kc1hSqpBaT0UUQPjAaW1OlGDVm1V7PjrUkdya64Rph9aw6GtNloGzRmiCNRctPc31tbi